#KPI Extraction BO

##Extraction of Token

In [0]:
fichero_reportes_a_procesar = "details_clusers_9_260204.csv"
entorno = "prod"
usuario = "user"
password="password"
delay=10
grupo="cluster_9_Invoicing"
range_initial=0
range_final=len(group_3)
fichero_documento_variables="variables_"+grupo+"_"+str(range_initial)+"_"+str(range_final)+".csv"
fichero_audit="audit_variables_"+grupo+"_"+str(range_initial)+"_"+str(range_final)+".csv"
fichero_audit_variables="audit_documento_variables_definicion_"+grupo+"_"+str(range_initial)+"_"+str(range_final)+".csv"
fichero_intermedio=grupo+"_variables_definition_"
fichero_definicion_variables=grupo+"_variables_definition_"+str(range_initial)+"_"+str(range_final)+".csv"
fichero_merged_definicion_variables = grupo +"_merged_variables_definition"+"_"+str(range_initial)+"_"+str(range_final)+".csv"
if entorno == "dev":
    url_logon ="https://bobidev.atradiusnet.com:8443/biprws/logon/long"
    url_logoff="https://bobidev.atradiusnet.com:8443/biprws/logoff"  
    url = "https://bobidev.atradiusnet.com:8443/biprws/raylight/v1/documents/"
if entorno == "prod":
    url_logon = "https://wavgbcdfp1088.atradiusnet.com:8443/biprws/logon/long"
    url_logoff = "https://wavgbcdfp1088.atradiusnet.com:8443/biprws/logoff" 
    url ="https://wavgbcdfp1088.atradiusnet.com:8443/biprws/raylight/v1/documents/"

In [0]:
import pandas as pd

group_3 = pd.read_csv(fichero_reportes_a_procesar, index_col=False)

group_3.drop(columns="Unnamed: 0", inplace=True)

In [0]:
display(group_3)

Group 1 and Group 5

In [0]:
len(group_3.Document_Id.unique())

In [0]:
import requests
def logon_request():
    payload = {"userName" : usuario,
    "password": password,
    "auth": "secLDAP"}


    headers = {
    "Content-Type": "application/json"
    }

    try:
        response = requests.post(url_logon, json=payload, headers=headers, verify=False, timeout=10)
        print("Status Code:", response.status_code)
        print("Response:", response)
        import xml.etree.ElementTree as ET

        # Parse the XML content
        root = ET.fromstring(response.content)

        # Define the namespace
        ns = {'atom': 'http://www.w3.org/2005/Atom', 'bip': 'http://www.sap.com/rws/bip'}

        # Extract the logonToken
        logonToken = root.find('.//bip:attr[@name="logonToken"]', ns).text

        print("Logon Token:")
        print(logonToken)
        return logonToken
    except requests.exceptions.ConnectionError as e:
        print("Connection Error: Cannot reach the server.")
        print("This hostname appears to be internal and not accessible from Databricks.")
        print(f"Error details: {e}")
    except Exception as e:
        print(f"Error: {e}")

In [0]:
import requests

def log_off_call(bo_url: str, logon_token: str):
   
        # Step 6: Log off session
        logoff_url = bo_url
        logoff_headers = {"X-SAP-LogonToken": logon_token}
        requests.post(logoff_url, headers=logoff_headers, verify=False)
        log_status: bool = False
        print("Logged off from BO REST API.")

## Extraction of document variables

In [0]:
logonToken=logon_request()

In [0]:
import xml.etree.ElementTree as ET
import pandas as pd
from time import sleep



variable_list = []
urls = []
http_code = [] 
variables = []
documentos = group_3.iloc[range_initial:range_final]["Document_Id"]
#logonToken=logon_request()
for DocumentId in documentos:
    url_request=url+str(DocumentId)+"/variables"
    payload = ""
    headers = {
    'X-SAP-LogonToken': logonToken,
    'Content-Type': 'application/json'}
  
    try:
        sleep(delay)
        print("url_request ",url_request)
        urls.append(url_request)
        variables.append(str(DocumentId))
        response = requests.get(url_request, headers=headers, verify=False, timeout=100)
        # Parse the XML response
        http_code.append(response.status_code)
        root = ET.fromstring(response.content)

        # Extract all variables
        variables_found = root.findall('.//variable')

        print(f"Found {len(variables_found)} variable(s)\n")

        # Parse each variable into a structured format
        
        for var in variables_found:
            var_data = {
                'documentId': DocumentId,
                'id': var.find('id').text if var.find('id') is not None else None,
                'name': var.find('name').text if var.find('name') is not None else None,
                'dataType': var.get('dataType'),
                'qualification': var.get('qualification'),
                'constant': var.get('constant'),
                'customSort': var.get('customSort'),
                'allowUserValues': var.get('allowUserValues'),
                'highPrecision': var.get('highPrecision'),
            }
            variable_list.append(var_data)

        # Create DataFrame for better visualization
        
        
        print("Status Code:", response.status_code)
        print("Response:", response.content)
    except requests.exceptions.ConnectionError as e:
        print("Connection Error: Cannot reach the server.")
        print("This hostname appears to be internal and not accessible from Databricks.")
        print(f"Error details: {e}")
    except Exception as e:
        print(f"Error: {e}")
df_variables = pd.DataFrame(variable_list)

In [0]:
audit = pd.DataFrame(data=zip(urls, http_code, variables), columns=['url','HTTPcode','Document_Id'])
audit.to_csv(fichero_audit)

In [0]:
df_variables.to_csv(fichero_documento_variables)

In [0]:
df_variables

In [0]:
logonToken

In [0]:
url_logoff

In [0]:

payload = {"userName" : usuario,
"password": password,
"auth": "secLDAP"}


headers = {
"Content-Type": "application/json"
}

log_off_call(url_logoff, logonToken)


##Get all the details for each variables of the reports

In [0]:
import pandas as pd
variables_group_3 = pd.read_csv(fichero_documento_variables)

In [0]:
variables_group_3

###Split of the requests in 500

It seems that event the qualification is different of a measure there are calculations

In [0]:
import xml.etree.ElementTree as ET
import pandas as pd
from time import sleep
#logon_Token = dbutils.secrets.get(scope = "bipr", key = "logonToken")
logonToken = logon_request()
# List to store all variable details
all_variables_data = []
Document_Ids=[]
variables=[]
http_code=[]

if len(variables_group_3)==0:
  print("No variables to fetch")
  log_off_call(url_logoff, logonToken)
else:
    i=0
    for index in range(0,len(variables_group_3['documentId']) ): 

            DocumentId = variables_group_3['documentId'][index]
            variable = variables_group_3['id'][index]
            url_request = url + str(DocumentId) + "/variables/" + str(variable) + ""
            print("url ",url_request)
            headers = {
                'X-SAP-LogonToken': logonToken,
                'Content-Type': 'application/json'
            }
            
            try:
                response = requests.get(url_request, headers=headers, verify=False, timeout=100)
                Document_Ids.append(DocumentId)
                variables.append(variable)
                http_code.append(response.status_code)
                print("Status Code:", response.status_code)
                print ('DocumentId ', DocumentId)
                print ('Variable ', variable)
                print("Num variable ",i,"/2872")
                print("=========")
                sleep(delay)
                i=i+1
                if response.status_code == 200:
                    # Parse the XML response
                    root = ET.fromstring(response.content)
                    
                    # Extract all relevant information
                    var_detail = {
                        'documentId': DocumentId,
                        'id': root.find('id').text if root.find('id') is not None else None,
                        'name': root.find('name').text if root.find('name') is not None else None,
                        'formulaLanguageId': root.find('formulaLanguageId').text if root.find('formulaLanguageId') is not None else None,
                        'definition': root.find('definition').text if root.find('definition') is not None else None,
                        'dataType': root.get('dataType'),
                        'qualification': root.get('qualification'),
                        'constant': root.get('constant'),
                        'highPrecision': root.get('highPrecision'),
                        'customSort': root.get('customSort'),
                        'stripped': root.get('stripped'),
                        'dataSourceEnriched': root.get('dataSourceEnriched')
                    }
                    all_variables_data.append(var_detail)
                    print(f"✓ Fetched: {variable}")
                else:
                    print(f"✗ Failed for {variable}: Status {response.status_code}")
                    
            except Exception as e:
                print(f"✗ Error for {variable}: {e}")
            
            sleep(0.5)  # Small delay to avoid overwhelming the server
            if index % 500 == 0:
                intermediate_save = pd.DataFrame(all_variables_data)
                intermediate_save.to_csv(fichero_intermedio+"_intermedio_"+str(index)+".csv", index=False)
# Create DataFrame with all variable details
df_all_variables = pd.DataFrame(all_variables_data)

print(f"\n\nTotal variables fetched: {len(df_all_variables)}")
log_off_call(url_logoff, logonToken)

In [0]:
audit_variables=pd.DataFrame(http_code,Document_Ids)
if audit_variables.shape[0]==0:
  print("No variables to fetch")
else:
    display(audit_variables)
    audit_variables.to_csv(fichero_audit_variables)
   

In [0]:
df_all_variables.to_csv(fichero_definicion_variables)

In [0]:
log_off_call(url_logoff, logonToken)

In [0]:
logonToken

In [0]:
df_all_variables.head()

In [0]:
df_variables = pd.read_csv(fichero_documento_variables)
df_variables.head()

In [0]:
df_merged = pd.merge(
        df_variables, 
        df_all_variables, 
        on=['documentId', 'id'], 
        how='inner',
        suffixes=('_basic', '_detailed')
    )

print(f"Merged DataFrame shape: {df_merged.shape}")
print(f"Columns: {list(df_merged.columns)}")
df_merged.to_csv(fichero_merged_definicion_variables)

display(df_merged)

In [0]:
fichero_merged_definicion_variables